### Imports, Parameters 

In [1]:
import numpy as np
import os
import subprocess
import matplotlib.pyplot as plt
from ase.io import read
from ase.build import bulk, make_supercell
import warnings
warnings.filterwarnings('ignore')

import ase
print(f'ASE version : {ase.__version__}')
print(f'NumPy       : {np.__version__}')

# ═══════════════════════════════════════════════════════════
# YOUR SYSTEM PATHS
# ═══════════════════════════════════════════════════════════
LAMMPS_CMD   = '/projects/westgroup/akinyemi.az/mace_lammps/lammps/build-mliap/lmp'
MACE_MODEL   = '/projects/westgroup/akinyemi.az/mace_lammps/models/mace-mp-0b2-medium.model-mliap_lammps.pt'
KOKKOS_FLAGS = ['-k', 'on', 'g', '1', '-sf', 'kk',
                '-pk', 'kokkos', 'newton', 'on', 'neigh', 'half']
SLURM_CONFIG = {
    'partition'     : 'multigpu',
    'ntasks'        : 1,
    'cpus_per_task' : 8,
    'gpu'           : 'a100:1',
    'time'          : '24:00:00',
    'conda_env'     : 'mace-lammps',
    'cuda_version'  : '12.3.0',
    'openmpi_ver'   : '4.1.6',
    'ld_paths'      : [
        '/shared/EL9/explorer/cuda/12.3.0/lib64/stubs',
        '/projects/westgroup/akinyemi.az/mace_lammps/lammps/build-mliap',
        '/home/akinyemi.az/miniforge3/envs/mace-lammps/lib',
    ]
}

# ── Fixed constants ──────────────────────────────────────────
ELEM_STR    = 'Al B C Cr Fe Mo Ni O H'
PAIR_STYLE  = 'mliap unified'
PAIR_SUFFIX = '0'
E2T = {'Al':1,'B':2,'C':3,'Cr':4,'Fe':5,'Mo':6,'Ni':7, 'O':8, 'H':9}
MASSES = {
    1:(26.9815,'Al'), 2:(10.8110,'B'),  3:(12.0110,'C'),
    4:(51.9961,'Cr'), 5:(55.8450,'Fe'), 6:(95.9600,'Mo'),
    7:(58.6934,'Ni'), 8:(15.9990, 'O'), 9:( 1.0080,'H'),
}

# ── MD parameters ────────────────────────────────────────────
TEMPERATURES  = [300, 400, 600, 800]  # K — Arrhenius points
TIMESTEP      = 0.0005   # ps = 0.5 fs
N_EQUIL       = 500000   # steps equilibration = 50 ps
N_PROD        = 1500000   # steps production = 100 ps (written for NB11)1620000
# N_PROD        = 1620000   # steps production = 100 ps (written for NB11)
TAU_T         = 0.1      # ps — Nosé-Hoover thermostat time constant
THERMO_EVERY  = 1000     # output frequency
DUMP_EVERY    = 1000      # trajectory dump frequency for MSD
# N_EQUIL = 1000000
# N_PROD = 200000
# DUMP_EVERY = 100

# ── File paths ───────────────────────────────────────────────
BULK_MIN  ='structures/02_bulk_energy_minimization/Hastelloy_N_7_minimized.lammps'

for d in ['structures/notebook10-bulk-equilibration/N_7', 'results/notebook10-bulk-equilibration/N_7', 'lammps_scripts/notebook10-bulk-equilibration/N_7', 'slurm_scripts/notebook10-bulk-equilibration/N_7']:
    os.makedirs(d, exist_ok=True)

# ── Read a₀ from NB02 ─────────────────────────────────────────
# Use minimized bulk structure to get correct lattice parameter
if not os.path.exists(BULK_MIN):
    raise FileNotFoundError(f'{BULK_MIN} not found. Complete NB02 first.')

bulk_slab = read(BULK_MIN, format='lammps-data', style='atomic')
A0 = bulk_slab.cell.lengths()[0] / 5  # 5x5x5 supercell → divide by 5
print(f'a₀ from minimized bulk: {A0:.6f} Å  (NB02)')

# # ── Read best subsurface site type from NB08 ─────────────────
# BEST_SUB_LABEL = 'O00'  # default fallback
# if os.path.exists('results/notebook08/subsurface_energies.txt'):
#     with open('results/notebook08/subsurface_energies.txt') as f:
#         for line in f:
#             if 'Most stable:' in line:
#                 BEST_SUB_LABEL = line.split()[2]
#                 break
# print(f'Best subsurface site from NB08: {BEST_SUB_LABEL}')
# print(f'Site type: octahedral — H at body-center of FCC unit cell')
# print(f'\nMD setup:')
# print(f'  Temperatures  : {TEMPERATURES} K')
# print(f'  Timestep      : {TIMESTEP*1000:.0f} fs')
# print(f'  Equilibration : {N_EQUIL*TIMESTEP:.0f} ps')
# print(f'  Production    : {N_PROD*TIMESTEP:.0f} ps')
# print(f'  Thermostat    : Nosé-Hoover τ = {TAU_T} ps')

ASE version : 3.27.0
NumPy       : 2.4.3
a₀ from minimized bulk: 3.572369 Å  (NB02)


In [ ]:
# # ─────────────────────────────────────────────────────────────
# # Build bulk supercell + 1 H at octahedral interstitial
# # Octahedral site in FCC: body center of unit cell = (½,½,½) frac
# # We place H near the geometric center of the supercell
# # Refs: H in FCC [3,4]
# # ─────────────────────────────────────────────────────────────

# bulk_atoms = read(BULK_MIN, format='lammps-data', style='atomic')
# bulk_pos   = bulk_atoms.get_positions()
# bulk_syms  = np.array(bulk_atoms.get_chemical_symbols())
# L          = bulk_atoms.cell.lengths()

# print(f'Bulk supercell: {len(bulk_atoms)} atoms')
# print(f'Cell: {L[0]:.4f} × {L[1]:.4f} × {L[2]:.4f} Å')
# print(f'a₀ = {L[0]/5:.6f} Å')

# # Find octahedral site near center of supercell
# # FCC octahedral site at fractional (½,½,½) within each unit cell
# # In the 5x5x5 supercell, the central unit cell is at (1,1,1) in cell coords
# # so the central octahedral site is at fractional (1.5/5, 1.5/5, 1.5/3) = (0.3, 0.3, 0.3)
# # i.e. the exact center of the supercell
# cx = L[0] / 2
# cy = L[1] / 2
# cz = L[2] / 2

# # Verify no metal atom is too close (overlap check)
# h_pos = np.array([cx, cy, cz])
# dists = np.linalg.norm(bulk_pos - h_pos, axis=1)
# min_dist = dists.min()
# print(f'\nH placed at ({cx:.4f}, {cy:.4f}, {cz:.4f}) Å')
# print(f'Nearest metal atom: {min_dist:.4f} Å  (should be > 1.5 Å)')

# if min_dist < 1.5:
#     # Shift H slightly if too close
#     h_pos = np.array([cx + 0.1, cy + 0.1, cz + 0.1])
#     dists = np.linalg.norm(bulk_pos - h_pos, axis=1)
#     min_dist = dists.min()
#     print(f'Shifted H to avoid overlap: min dist = {min_dist:.4f} Å')

# # Build combined structure
# all_syms = list(bulk_syms) + ['H']
# all_pos  = np.vstack([bulk_pos, h_pos])

# # Write LAMMPS data file
# BULK_H_FILE = 'structures/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_H_initial.lammps'
# with open(BULK_H_FILE, 'w') as f:
#     f.write(f'# Hastelloy N bulk 5x5x5 + 1H at octahedral site — NB10\n\n')
#     f.write(f'{len(all_syms)} atoms\n{len(MASSES)} atom types\n\n')
#     f.write(f'0.0  {L[0]:.10f}  xlo xhi\n')
#     f.write(f'0.0  {L[1]:.10f}  ylo yhi\n')
#     f.write(f'0.0  {L[2]:.10f}  zlo zhi\n\n')
#     f.write('Masses\n\n')
#     for t, (m, el) in MASSES.items():
#         f.write(f'{t}  {m}  # {el}\n')
#     f.write('\nAtoms # atomic\n\n')
#     for i, (sym, p) in enumerate(zip(all_syms, all_pos), 1):
#         f.write(f'{i}  {E2T[sym]}  {p[0]:.10f}  {p[1]:.10f}  {p[2]:.10f}\n')

# print(f'\nWritten: {BULK_H_FILE}')
# print(f'  Total atoms : {len(all_syms)} (500 metal + 1 H)')
# print(f'  H at        : ({h_pos[0]:.4f}, {h_pos[1]:.4f}, {h_pos[2]:.4f}) Å')
# print(f'  c_H         : {1/500*100:.2f} at.%  (dilute limit)')

Bulk supercell: 500 atoms
Cell: 17.8618 × 17.8618 × 17.8618 Å
a₀ = 3.572369 Å

H placed at (8.9309, 8.9309, 8.9309) Å
Nearest metal atom: 1.6504 Å  (should be > 1.5 Å)

Written: structures/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_H_initial.lammps
  Total atoms : 501 (500 metal + 1 H)
  H at        : (8.9309, 8.9309, 8.9309) Å
  c_H         : 0.20 at.%  (dilute limit)


In [ ]:
# ── H insertion parameters  ─────────────────────────────────
N_H         = 10    # ← set number of H atoms to insert (exact integer)
D_MIN_METAL = 1.5   # Å — minimum H-to-metal distance
D_MIN_HH    = 2.5   # Å — minimum H-to-H distance
SEED_H      = 42    # random seed for reproducible site selection

In [ ]:
print(f'a₀ from minimized bulk: {A0:.6f} Å  (NB02)')
print(f'H insertion : N_H={N_H}, D_MIN_METAL={D_MIN_METAL} Å, D_MIN_HH={D_MIN_HH} Å, seed={SEED_H}')

In [ ]:
def insert_hydrogen(
    bulk_min_path,
    n_h,
    masses,
    e2t,
    out_dir,
    a0,
    supercell_reps=(5, 5, 5),
    d_min_metal=1.5,
    d_min_hh=2.5,
    seed=42,
):
    """
    Insert H atoms at FCC octahedral interstitial sites in a bulk supercell
    and write a LAMMPS data file.

    Octahedral sites in FCC lie at the body-centre of each unit cell,
    i.e. fractional coordinate (1/2, 1/2, 1/2).  All candidate sites are
    enumerated from the lattice parameter, randomly shuffled, and
    greedily accepted subject to two distance thresholds.

    Parameters
    ----------
    bulk_min_path : str
        Path to the energy-minimized LAMMPS data file (lammps-data
        atomic style) produced by NB02.
    n_h : int
        Number of H atoms to insert.
    masses : dict
        Mapping {atom_type: (mass, element_symbol)} — all element types
        must be present so the LAMMPS header is complete.
    e2t : dict
        Mapping {element_symbol: atom_type} used to assign integer
        type IDs in the output file.
    out_dir : str
        Directory in which the output LAMMPS data file is written.
    a0 : float
        Lattice parameter in Angstrom, used to compute octahedral site
        coordinates.
    supercell_reps : tuple of int, optional
        Repetitions along (x, y, z) used when building the supercell
        in NB01/NB02.  Default is (5, 5, 5).
    d_min_metal : float, optional
        Minimum allowed distance (Angstrom) between a candidate H site
        and any metal atom.  Sites closer than this are rejected.
        Default is 1.5.
    d_min_hh : float, optional
        Minimum allowed distance (Angstrom) between two H atoms.
        Enforces the dilute-limit approximation by preventing clustering.
        Default is 2.5.
    seed : int, optional
        NumPy random seed for reproducible site shuffling.
        Default is 42.

    Returns
    -------
    out_path : str
        Absolute path to the written LAMMPS data file.
    h_positions : numpy.ndarray, shape (n_h, 3)
        Cartesian coordinates (Angstrom) of the accepted H sites.
    min_hm_dist : float
        Smallest distance (Angstrom) between any H atom and its nearest
        metal neighbour — useful as a post-insertion sanity check.

    Raises
    ------
    FileNotFoundError
        If ``bulk_min_path`` does not exist.
    RuntimeError
        If fewer than ``n_h`` valid octahedral sites can be found given
        the supplied distance thresholds.
    """
    if not os.path.exists(bulk_min_path):
        raise FileNotFoundError(f'{bulk_min_path} not found.')

    # ── Load minimized bulk ───────────────────────────────────
    bulk_atoms = read(bulk_min_path, format='lammps-data', style='atomic')
    bulk_pos   = bulk_atoms.get_positions()           # (N_metal, 3)
    bulk_syms  = np.array(bulk_atoms.get_chemical_symbols())
    L          = bulk_atoms.cell.lengths()

    nx, ny, nz = supercell_reps
    print(f'Bulk supercell : {len(bulk_atoms)} atoms')
    print(f'Cell           : {L[0]:.4f} x {L[1]:.4f} x {L[2]:.4f} Å')
    print(f'a₀             : {a0:.6f} Å')

    # ── Enumerate all octahedral sites ────────────────────────
    # Body-centre of each unit cell: r_ijk = ((i+0.5)*a0, (j+0.5)*a0, (k+0.5)*a0)
    candidates = np.array([
        [(i + 0.5) * a0, (j + 0.5) * a0, (k + 0.5) * a0]
        for i in range(nx)
        for j in range(ny)
        for k in range(nz)
    ])                                                 # (nx*ny*nz, 3)
    print(f'Octahedral candidates : {len(candidates)}')

    # ── Shuffle for random selection ──────────────────────────
    rng = np.random.default_rng(seed)
    rng.shuffle(candidates)

    # ── Greedy acceptance with dual distance check ────────────
    accepted = []   # list of accepted position arrays

    for site in candidates:
        if len(accepted) == n_h:
            break

        # Check 1: distance to all metal atoms
        if np.linalg.norm(bulk_pos - site, axis=1).min() < d_min_metal:
            continue

        # Check 2: distance to already-accepted H atoms
        if accepted:
            if np.linalg.norm(np.array(accepted) - site, axis=1).min() < d_min_hh:
                continue

        accepted.append(site)

    if len(accepted) < n_h:
        raise RuntimeError(
            f'Only {len(accepted)}/{n_h} valid sites found. '
            f'Try reducing n_h, d_min_metal ({d_min_metal} Å), '
            f'or d_min_hh ({d_min_hh} Å).'
        )

    h_positions = np.array(accepted)                  # (n_h, 3)

    # ── Nearest metal–H distance across all inserted H ────────
    all_hm = np.linalg.norm(
        bulk_pos[:, None, :] - h_positions[None, :, :], axis=2
    )                                                  # (N_metal, n_h)
    min_hm_dist = float(all_hm.min())

    # ── Build combined atom lists ──────────────────────────────
    all_syms = list(bulk_syms) + ['H'] * n_h
    all_pos  = np.vstack([bulk_pos, h_positions])

    # ── Write LAMMPS data file ─────────────────────────────────
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f'Hastelloy_N_7_{n_h}H_initial.lammps')
    with open(out_path, 'w') as f:
        f.write(f'# Hastelloy N bulk 5x5x5 + {n_h}H at octahedral sites — NB10\n\n')
        f.write(f'{len(all_syms)} atoms\n{len(masses)} atom types\n\n')
        f.write(f'0.0  {L[0]:.10f}  xlo xhi\n')
        f.write(f'0.0  {L[1]:.10f}  ylo yhi\n')
        f.write(f'0.0  {L[2]:.10f}  zlo zhi\n\n')
        f.write('Masses\n\n')
        for t, (m, el) in masses.items():
            f.write(f'{t}  {m}  # {el}\n')
        f.write('\nAtoms # atomic\n\n')
        for i, (sym, p) in enumerate(zip(all_syms, all_pos), 1):
            f.write(f'{i}  {e2t[sym]}  {p[0]:.10f}  {p[1]:.10f}  {p[2]:.10f}\n')

    return out_path, h_positions, min_hm_dist


# ── Run ───────────────────────────────────────────────────────
OUT_DIR     = 'structures/notebook10-bulk-equilibration/N_7'
BULK_H_FILE, h_pos_all, min_hm = insert_hydrogen(
    bulk_min_path  = BULK_MIN,
    n_h            = N_H,
    masses         = MASSES,
    e2t            = E2T,
    out_dir        = OUT_DIR,
    a0             = A0,
    supercell_reps = (5, 5, 5),
    d_min_metal    = D_MIN_METAL,
    d_min_hh       = D_MIN_HH,
    seed           = SEED_H,
)

n_metal = 500
print(f'\nWritten : {BULK_H_FILE}')
print(f'  Total atoms       : {n_metal + N_H}  ({n_metal} metal + {N_H} H)')
print(f'  Nearest metal-H   : {min_hm:.4f} Å  (threshold {D_MIN_METAL} Å)')
print(f'  c_H               : {N_H / n_metal * 100:.2f} at.%')
print(f'\nAccepted H positions (Å):')
for idx, hp in enumerate(h_pos_all, 1):
    print(f'  H{idx:>3d}  ({hp[0]:.4f}, {hp[1]:.4f}, {hp[2]:.4f})')

In [4]:
# ─────────────────────────────────────────────────────────────
# Write LAMMPS NVT script for bulk H equilibration
# Phase 1: equilibration (50 ps) — system reaches T
# Phase 2: production (100 ps) — trajectory written for NB11 MSD
# Nosé-Hoover thermostat on all atoms (no frozen layers in bulk)
# Refs: NVT MD [6]; H diffusion [1,2]
# ─────────────────────────────────────────────────────────────

def write_bulk_nvt(T):
    traj_file = f'results/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_H_{T}K_prod.lammpstrj'
    log_file  = f'results/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_equil_{T}K.log'
    out_file  = f'structures/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_H_{T}K.lammps'

    script = f"""# LAMMPS NVT bulk equilibration — Hastelloy N + 1H
# T = {T} K | Notebook 10
# Phase 1: {N_EQUIL*TIMESTEP:.0f} ps equilibration
# Phase 2: {N_PROD*TIMESTEP:.0f} ps production (trajectory for NB11 MSD)
# Refs: NVT [6]; H diffusion [1,2]

units          metal
atom_style     atomic
newton         on
boundary       p p p

read_data      {BULK_H_FILE}

pair_style     {PAIR_STYLE} {MACE_MODEL} {PAIR_SUFFIX}
pair_coeff     * * {ELEM_STR}

neighbor       2.0 bin
neigh_modify   every 1 delay 0 check yes

# ── Identify H atom for tracking ─────────────────────────────
group          H_atom   type 9
group          metal    subtract all H_atom

# ── Thermo ───────────────────────────────────────────────────
thermo         {THERMO_EVERY}
thermo_style   custom step temp pe ke etotal press vol

# ── Timestep ─────────────────────────────────────────────────
timestep       {TIMESTEP}

# ── Initialize velocities ────────────────────────────────────
velocity       all create {T}.0 42 mom yes rot yes dist gaussian

# ── Phase 1: Equilibration ───────────────────────────────────
fix            nvt_equil  all  nvt  temp  {T}.0  {T}.0  {TAU_T}
print "### Phase 1: Equilibration {N_EQUIL*TIMESTEP:.0f} ps at {T} K ###"
run            {N_EQUIL}
unfix          nvt_equil

# ── Phase 2: Production ──────────────────────────────────────
# Dump trajectory for NB11 MSD calculation
# Dump every {DUMP_EVERY} steps = {DUMP_EVERY*TIMESTEP*1000:.0f} fs
dump           prod_dump  all  custom  {DUMP_EVERY}  {traj_file} &
               id type x y z
dump_modify    prod_dump  sort id

fix            nvt_prod  all  nvt  temp  {T}.0  {T}.0  {TAU_T}

# Compute MSD for H atom
compute        msd_H  H_atom  msd
fix            msd_out  all  ave/time  1  1  {THERMO_EVERY} &
               c_msd_H[4]  file  results/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_msd_H_{T}K.txt  mode scalar

print "### Phase 2: Production {N_PROD*TIMESTEP:.0f} ps at {T} K ###"
run            {N_PROD}

# ── Save final state ─────────────────────────────────────────
variable  pe_final  equal  pe
variable  temp_fin  equal  temp
variable  press_fin equal  press

print "EQUIL_RESULTS_START"
print "  T_K          : {T}"
print "  pe_final_eV  : ${{pe_final}}"
print "  temp_final_K : ${{temp_fin}}"
print "  press_final  : ${{press_fin}}"
print "EQUIL_RESULTS_END"

write_data     {out_file}
"""
    path = f'lammps_scripts/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_nvt_{T}K.lammps'
    with open(path, 'w') as f:
        f.write(script)
    return path, log_file, out_file, traj_file

job_list = []
sc  = SLURM_CONFIG
kk  = ' '.join(KOKKOS_FLAGS)
ld  = '\n'.join(f'export LD_LIBRARY_PATH={p}:$LD_LIBRARY_PATH'
                for p in sc['ld_paths'])

for T in TEMPERATURES:
    script_f, log_f, out_f, traj_f = write_bulk_nvt(T)

    slurm = f"""#!/bin/bash
#SBATCH --job-name=Hastelloy_N_7equilibration_{T}K
#SBATCH --ntasks={sc['ntasks']}
#SBATCH --cpus-per-task={sc['cpus_per_task']}
#SBATCH --gres=gpu:{sc['gpu']}
#SBATCH --partition={sc['partition']}
#SBATCH --time={sc['time']}
#SBATCH --output=results/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_slurm_bulk_{T}K_%j.out

module load OpenMPI/{sc['openmpi_ver']}
module load cuda/{sc['cuda_version']}
source ~/miniforge3/etc/profile.d/conda.sh
conda activate {sc['conda_env']}
{ld}
cd {os.getcwd()}

echo "T={T}K Start: $(date)"
{LAMMPS_CMD} {kk} -in {script_f} -log {log_f}
echo "T={T}K End: $(date)"
"""
    sp = f'slurm_scripts/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_run_bulk_{T}K.sh'
    with open(sp, 'w') as f:
        f.write(slurm)
    os.chmod(sp, 0o755)
    job_list.append((T, script_f, log_f, out_f, traj_f, sp))

print(f'Written {len(job_list)} LAMMPS + SLURM scripts')
for T, sf, lf, of, tf, sp in job_list:
    print(f'  {T} K: {sf}')
    print(f'        traj → {tf}')

Written 4 LAMMPS + SLURM scripts
  300 K: lammps_scripts/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_nvt_300K.lammps
        traj → results/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_H_300K_prod.lammpstrj
  400 K: lammps_scripts/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_nvt_400K.lammps
        traj → results/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_H_400K_prod.lammpstrj
  600 K: lammps_scripts/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_nvt_600K.lammps
        traj → results/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_H_600K_prod.lammpstrj
  800 K: lammps_scripts/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_nvt_800K.lammps
        traj → results/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_H_800K_prod.lammpstrj


In [2]:
# ─────────────────────────────────────────────────────────────
# Submit all 4 temperature jobs simultaneously
# 4 jobs × 1 A100 each — well within cluster limits
# All jobs are independent — no inter-job communication needed
# ─────────────────────────────────────────────────────────────

job_ids = {}

print(f'Submitting {len(job_list)} NVT jobs...')
print(f'{"T (K)":>8s}  {"Job ID"}')
print('-' * 25)

for T, sf, lf, of, tf, sp in job_list:
    result = subprocess.run(['sbatch', sp], capture_output=True, text=True)
    if result.returncode == 0:
        jid = result.stdout.strip().split()[-1]
        job_ids[T] = jid
        print(f'  {T:>6d} K  → Job {jid}')
    else:
        print(f'  {T:>6d} K  FAILED: {result.stderr.strip()[:50]}')

print(f'\n{len(job_ids)}/4 jobs submitted.')
print('Monitor: squeue -u $USER')
print(f'Each job: {(N_EQUIL+N_PROD)*TIMESTEP:.0f} ps total ({(N_EQUIL+N_PROD)*TIMESTEP/1000:.3f} ns)')
print('After all jobs finish, run cell 3.5.')

NameError: name 'job_list' is not defined

In [3]:
# ─────────────────────────────────────────────────────────────
# Parse equilibration logs
# Verify: T reached, PE stable, no crashes
# ─────────────────────────────────────────────────────────────

def parse_equil_log(logfile):
    """Extract final T, PE, pressure from EQUIL_RESULTS block."""
    results = {}
    if not os.path.exists(logfile):
        return None
    with open(logfile) as f:
        for line in f:
            line = line.strip()
            for key in ['T_K', 'pe_final_eV', 'temp_final_K', 'press_final']:
                if key in line and ':' in line:
                    try:
                        results[key] = float(line.split(':')[1].strip())
                    except:
                        pass
    return results if results else None

def parse_thermo_series(logfile):
    """Read LAMMPS thermo output — return step, temp, pe arrays."""
    steps, temps, pes = [], [], []
    in_thermo = False
    if not os.path.exists(logfile):
        return None
    with open(logfile) as f:
        for line in f:
            line = line.strip()
            #if line.startswith('Step') and 'Temp' in line and 'PotEng' in line:
            if line.startswith('Step') and 'Temp' in line:
                in_thermo = True
                continue
            if in_thermo:
                if line.startswith('Loop') or line.startswith('WARNING'):
                    in_thermo = False
                    continue
                parts = line.split()
                if len(parts) >= 3:
                    try:
                        steps.append(float(parts[0]))
                        temps.append(float(parts[1]))
                        pes.append(float(parts[2]))
                    except:
                        pass
    if steps:
        return np.array(steps), np.array(temps), np.array(pes)
    return None

# Rebuild job_list if kernel restarted
job_list = []
for T in TEMPERATURES:
    job_list.append((
        T,
        f'lammps_scripts/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_nvt_{T}K.lammps',
        f'results/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_equil_{T}K.log',
        f'structures/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_H_{T}K.lammps',
        f'results/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_H_{T}K_prod.lammpstrj',
        f'slurm_scripts/notebook10-bulk-equilibration/N_7/run_Hastelloy_N_7_{T}K.sh'
    ))

print('Bulk Equilibration Results')
print('=' * 70)
print(f'{"T (K)":>6s} {"T_final (K)":>12s} {"PE (eV)":>14s} '
      f'{"Traj exists":>12s} {"Status"}')
print('-' * 70)

summary = {}
missing = []
all_ok  = True

for T, sf, lf, of, tf, sp in job_list:
    res = parse_equil_log(lf)
    traj_ok = os.path.exists(tf) and os.path.getsize(tf) > 0
    struct_ok = os.path.exists(of)

    if res:
        T_fin = res.get('temp_final_K', res.get('T_K', 0))
        pe    = res.get('pe_final_eV', 0)
        T_ok  = abs(T_fin - T) < 50  # within 50 K
        ok    = T_ok and traj_ok and struct_ok
        summary[T] = {'T_final': T_fin, 'pe': pe, 'ok': ok}
        status = '✓' if ok else 'WARN'
        print(f'  {T:>6d} {T_fin:>12.1f} {pe:>14.4f} '
              f'{"yes" if traj_ok else "NO":>12s} {status}')
        if not ok:
            all_ok = False
    else:
        missing.append(T)
        print(f'  {T:>6d} {"NOT FOUND":>12s}')
        all_ok = False

print('=' * 70)
if missing:
    print(f'⚠ {len(missing)} jobs not done: {missing}')
elif all_ok:
    print('All 4 temperatures equilibrated successfully.')

# Save summary
if summary:
    with open('results/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_equil_summary.txt', 'w') as f:
        f.write('Hastelloy N Bulk Equilibration — Notebook 10\n')
        f.write('=' * 50 + '\n')
        f.write(f'System   : 500 metal + 1 H (octahedral site)\n')
        f.write(f'Protocol : {N_EQUIL*TIMESTEP:.0f} ps equil + {N_PROD*TIMESTEP:.0f} ps production\n\n')
        f.write(f'{"T_set":>8s} {"T_final":>10s} {"PE (eV)":>14s} {"OK"}\n')
        f.write('-' * 40 + '\n')
        for T, d in sorted(summary.items()):
            f.write(f'{T:>8d} {d["T_final"]:>10.1f} {d["pe"]:>14.4f} '
                    f'{"yes" if d["ok"] else "no"}\n')
    print('Saved: results/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_equil_summary.txt')

Bulk Equilibration Results
 T (K)  T_final (K)        PE (eV)  Traj exists Status
----------------------------------------------------------------------
     300        287.1     -3353.7835          yes ✓
     400        372.7     -3351.0484          yes ✓
     600        593.4     -3336.7312          yes ✓
     800        795.9     -3329.5882          yes ✓
All 4 temperatures equilibrated successfully.
Saved: results/notebook10-bulk-equilibration/N_7/Hastelloy_N_7_equil_summary.txt
